<a href="https://colab.research.google.com/github/Janusha-Weerasinghe/paddy_leaf_disease_detection_system-back-end/blob/dev/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow==2.19.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.3 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.18.0
    Uninstalling tensorflow-2.18.0:
      Successfully uninstalled tensorflow-2.18.0


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dedeikhsandwisaputra/rice-leafs-disease-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'rice-leafs-disease-dataset' dataset.
Path to dataset files: /kaggle/input/rice-leafs-disease-dataset


In [ ]:
# ---- IMPORTS ----
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
import kagglehub
from google.colab import files

In [ ]:
# ---- CONFIG ----
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100
MODEL_OUT = "/content/classify.keras"  # Use native Keras format (.keras)
BACKBONE = "mobilenet"   # or "efficientnet"


In [ ]:
# ---- DOWNLOAD DATASET ----
print("Downloading Kaggle dataset...")
DATASET_ROOT = kagglehub.dataset_download("dedeikhsandwisaputra/rice-leafs-disease-dataset")
print("Dataset root path:", DATASET_ROOT)


Using Colab cache for faster access to the 'rice-leafs-disease-dataset' dataset.
Dataset root path: /kaggle/input/rice-leafs-disease-dataset


In [ ]:
# The actual images are inside the 'RiceLeafsDisease' folder
DATA_PATH = os.path.join(DATASET_ROOT, "RiceLeafsDisease")
print("Using data path:", DATA_PATH)

# ---- DATASET ----
train_dir = os.path.join(DATA_PATH, "train")
val_dir = os.path.join(DATA_PATH, "validation")  # Make sure folder is 'validation'

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

# Get class names BEFORE prefetch
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Using data path: /kaggle/input/rice-leafs-disease-dataset/RiceLeafsDisease
Found 2100 files belonging to 6 classes.
Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot']
Found 528 files belonging to 6 classes.


In [ ]:
# ---- MODEL ----
def build_model(input_shape=(224,224,3), num_classes=2, backbone="mobilenet"):
    if backbone == "mobilenet":
        base = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    else:
        base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=base.input, outputs=out)
    return model

model = build_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=num_classes, backbone=BACKBONE)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# ---- CALLBACKS ----
callbacks = [
    ModelCheckpoint(MODEL_OUT, save_best_only=True, monitor="val_loss"),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    TensorBoard(log_dir="logs/classify")
]

# ---- TRAIN ----
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

Epoch 1/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 127s 2s/step - accuracy: 0.4405 - loss: 1.4250 - val_accuracy: 0.7254 - val_loss: 0.7457 - learning_rate: 0.0010
Epoch 2/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.6981 - loss: 0.8198 - val_accuracy: 0.7273 - val_loss: 0.6781 - learning_rate: 0.0010
Epoch 3/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.7151 - loss: 0.7010 - val_accuracy: 0.7538 - val_loss: 0.6373 - learning_rate: 0.0010
Epoch 4/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.7536 - loss: 0.6518 - val_accuracy: 0.7614 - val_loss: 0.6061 - learning_rate: 0.0010
Epoch 5/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.7678 - loss: 0.5800 - val_accuracy: 0.7595 - val_loss: 0.5939 - learning_rate: 0.0010
Epoch 6/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.7882 - loss: 0.5546 - val_accuracy: 0.7727 - val_loss: 0.5684 - learning_rate: 0.0010
Epoch 7/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.7904 - loss: 0.5473 - 

In [ ]:
# ---- SAVE MODEL & LABELS ----
model.save(MODEL_OUT)  # Native Keras format
labels_path = MODEL_OUT.replace(".keras", "_labels.txt")
with open(labels_path, "w") as f:
    for c in class_names:
        f.write(c + "\n")

print("✅ Model saved locally:", MODEL_OUT)
print("✅ Labels saved locally:", labels_path)

files.download(MODEL_OUT)
files.download(labels_path)

✅ Model saved locally: /content/classify.keras
✅ Labels saved locally: /content/classify_labels.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>